# scBaseCount Homo sapiens metadata (parquet)

This notebook works through the questions in `DailyNotes.md` (2026-03-23):

- What do the parquet columns represent, and how do `obs_metadata` vs `sample_metadata` relate?
- At Homo sapiens scale: how many SRX accessions (studies / processed experiments), and how many cells?
- For **cancer**-related records: which diseases appear, and how many distinct labels (subcategories)?
- For **immune-oncology (IO)**-relevant records: which SRX rows look IO-related via `perturbation` / `disease` / `tissue`, and how many cells and subcategories per slice?
- What is available about **patient-level** fields (e.g. cancer stage) vs what is **not** in these tables?
- **IO × disease-theme cohort cards:** the same workflow repeated per disease area (e.g. breast, lung, melanoma)—**cells + SRX counts only**, no patient rollups—plus top `disease` / `perturbation` lines for a narrative card.

**Data layout:** point `DATA_ROOT` at a scBasecount drop that contains `metadata/GeneFull/Homo_sapiens/*.parquet` (or adjust the glob below).

In [4]:
from __future__ import annotations

import re
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds

DATA_ROOT = (Path.cwd().parent / "data" / "scbasecount" / "2026-01-12").resolve()

META_GLOB = "**/metadata/**/Homo_sapiens/*Homo_sapiens*_metadata*.parquet"

paths = sorted(DATA_ROOT.glob(META_GLOB))
if not paths:
    raise FileNotFoundError(
        f"No Homo sapiens parquet under {DATA_ROOT!s}. Set DATA_ROOT to your scBaseCount metadata folder."
    )

obs_paths = [p for p in paths if "obs_metadata" in p.name]
sample_paths = [p for p in paths if "sample_metadata" in p.name]
assert len(obs_paths) == 1 and len(sample_paths) == 1, (obs_paths, sample_paths)
OBS_PARQUET = obs_paths[0]
SAMPLE_PARQUET = sample_paths[0]

SUMMARY_PATH = Path.cwd() / "scbasecount_homo_sapiens_metadata_summary.txt"


def _init_summary_file() -> None:
    SUMMARY_PATH.write_text(
        "scBaseCount Homo sapiens metadata run summary\n"
        f"written {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')} UTC\n"
        f"DATA_ROOT: {DATA_ROOT!s}\n"
        "---\n",
        encoding="utf-8",
    )


def summary_line(message: str = "", *, also_print: bool = True) -> None:
    if also_print:
        print(message)
    with SUMMARY_PATH.open("a", encoding="utf-8") as f:
        f.write(message + "\n")


_init_summary_file()
OBS_PARQUET, SAMPLE_PARQUET

(PosixPath('/Users/otodreas/Desktop/Work/Nygen/scBaseCount_Pipeline/data/scbasecount/2026-01-12/metadata/GeneFull/Homo_sapiens/scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_obs_metadata.parquet'),
 PosixPath('/Users/otodreas/Desktop/Work/Nygen/scBaseCount_Pipeline/data/scbasecount/2026-01-12/metadata/GeneFull/Homo_sapiens/scbasecount_2026-01-12_metadata_GeneFull_Homo_sapiens_sample_metadata.parquet'))

## 1. Column reference (what each file stores)

| File | Grain | Role |
|------|--------|------|
| `*_obs_metadata.parquet` | one row per **cell** | Barcodes, SRX accession, QC-ish gene/UMI counts, optional `cell_type` / `cell_ontology_term_id`. |
| `*_sample_metadata.parquet` | one row per **SRX** (processed experiment accession) | Study-level annotations Arc attached: tissue, disease, perturbation, `obs_count` (cells in that SRX), paths to `h5ad`, etc. |

Patient identifiers (e.g. individual donor counts, clinical stage) are **not** guaranteed in these columns; see the gap analysis section below.

In [5]:
obs_schema = pq.read_schema(OBS_PARQUET)
sample_schema = pq.read_schema(SAMPLE_PARQUET)

pd.DataFrame(
    {"obs_metadata": obs_schema.names, "dtype": [str(obs_schema.field(n).type) for n in obs_schema.names]}
)

,obs_metadata,dtype
0,cell_barcode,string
1,SRX_accession,string
2,gene_count_Unique,int64
3,umi_count_Unique,float
4,gene_count_UniqueAndMult-EM,int64
5,umi_count_UniqueAndMult-EM,float
6,gene_count_UniqueAndMult-Uniform,int64
7,umi_count_UniqueAndMult-Uniform,float
8,cell_type,string
9,cell_ontology_term_id,string


In [6]:
pd.DataFrame(
    {
        "sample_metadata": sample_schema.names,
        "dtype": [str(sample_schema.field(n).type) for n in sample_schema.names],
    }
)

,sample_metadata,dtype
0,entrez_id,int64
1,srx_accession,string
2,file_path,string
3,obs_count,int64
4,lib_prep,string
5,tech_10x,string
6,cell_prep,string
7,organism,string
8,tissue,string
9,tissue_ontology_term_id,string


## 2. Homo sapiens scale: SRX rows and cells

Use `sample_metadata.obs_count` summed for total cells (matches one row per cell in `obs_metadata` if the pipeline is consistent). Cross-check with parquet row count on `obs_metadata`.

In [7]:
# Number of cells = number of rows in obs_metadata
pf_obs = pq.ParquetFile(OBS_PARQUET)
n_cells_obs_file = pf_obs.metadata.num_rows
n_cells_obs_file

302277013

In [8]:
# Number of SRX = number of rows in sample_metadata
# Each SRX has a number of cells = obs_count
sample = pd.read_parquet(SAMPLE_PARQUET)
sample["obs_count"] = pd.to_numeric(sample["obs_count"], errors="coerce")

_UNKNOWN_LIKE_DISEASE = frozenset(
    {
        "",
        "unknown",
        "unsure",
        "not known",
        "not specified",
        "not applicable",
        "n/a",
        "na",
        "none",
        "null",
        "?",
        "-",
        "--",
        "undetermined",
        "unclear",
        "ambiguous",
    }
)


def _disease_is_unknown_like(d: pd.Series) -> pd.Series:
    t = d.fillna("").astype(str).str.strip().str.lower()
    return d.isna() | t.isin(_UNKNOWN_LIKE_DISEASE) | t.str.contains(r"n/?a", na=False)


_n_srx_before = len(sample)
sample = sample.loc[~_disease_is_unknown_like(sample["disease"])].copy()
summary_line(
    f"Excluded {_n_srx_before - len(sample):,} SRX rows with missing or unknown-like disease "
    f"(from {_n_srx_before:,} loaded)"
)

n_srx = sample["srx_accession"].nunique()
n_cells_from_sample = int(sample["obs_count"].sum())
summary_line(f"SRX rows (sample_metadata): {len(sample):,}")
summary_line(f"Unique srx_accession: {n_srx:,}")
summary_line(f"Sum of obs_count (cells, from sample table): {n_cells_from_sample:,}")
summary_line(f"obs_metadata parquet row count: {n_cells_obs_file:,}")

Excluded 12,735 SRX rows with missing or unknown-like disease (from 35,266 loaded)
SRX rows (sample_metadata): 22,531
Unique srx_accession: 22,531
Sum of obs_count (cells, from sample table): 180,529,353
obs_metadata parquet row count: 302,277,013


### Disease and tissue labels (exploratory)

Summaries below use `sample_metadata` **after dropping** SRX rows whose `disease` is missing or a placeholder (e.g. unknown, unsure, not specified). Counts are **weighted by `obs_count`** (cells) and include **how many SRX rows** carry each label. Use this to sanity-check regex **themes** in section 5b.

In [9]:
def summarize_labels(
    df: pd.DataFrame, col: str, weight: str = "obs_count", top_n: int = 40
) -> pd.DataFrame:
    w = pd.to_numeric(df[weight], errors="coerce").fillna(0)
    tmp = df.assign(_w=w)
    g = tmp.groupby(col, dropna=False)._w.agg(["sum", "count"])
    g = g.rename(columns={"sum": "cells", "count": "n_srx_rows"}).astype({"cells": "int64"})
    return g.sort_values("cells", ascending=False).head(top_n)


summary_line(
    f"disease: {sample['disease'].isna().sum():,} nulls, "
    f"{sample['disease'].nunique():,} distinct strings"
)
summary_line(
    f"tissue:  {sample['tissue'].isna().sum():,} nulls, "
    f"{sample['tissue'].nunique():,} distinct strings"
)

top_disease = summarize_labels(sample, "disease")
top_tissue = summarize_labels(sample, "tissue")

display(top_disease)
display(top_tissue)

disease: 0 nulls, 5,604 distinct strings
tissue:  0 nulls, 5,864 distinct strings


,cells,n_srx_rows
disease,,
chronic myelogenous leukemia,13421381,935
acute T cell leukemia,5550486,513
Chronic Myelogenous Leukemia,4336204,298
none reported,4146231,557
COVID-19,3966630,457
other,2886005,284
normal,2407983,349
healthy,2142169,272
breast cancer,1983288,205


,cells,n_srx_rows
tissue,,
CML,10841733,760
unsure,9017030,758
K562 cell line,8323474,580
blood,6480923,790
acute T cell leukemia,4021580,368
bone marrow,3897605,571
Peripheral Blood Mononuclear Cells (PBMC),3712623,369
Peripheral blood mononuclear cells (PBMCs),3483890,322
lung,3119580,454


## 3. Cancer-related slice: diseases and subcategory counts

"Subcategories" here means distinct `disease` strings after a broad cancer keyword filter (no ontology merge—raw labels as in the table).

In [10]:
CANCER_RE = re.compile(
    r"cancer|carcinoma|melanoma|lymphoma|leukemia|leukaemia|myeloma|sarcoma|tumor|tumour|neoplasm|oma\b|glioma|blastoma",
    re.I,
)

NORMAL_RE = re.compile(
    r"\b(?:normal|control(?:s)?|healthy|unaffected|wild[ -]?type|naive|wt|vehicle|sham|untreated|reference|non-disease|no disease|normal tissue|healthy donor|none reported|not specified|not_applicable|not applicable|none|noncancer|nondisease|non-diseased)\b",
    re.I,
)


cancer_mask = sample["disease"].str.contains(CANCER_RE, na=False)
normal_mask = sample["disease"].str.contains(NORMAL_RE, na=False)
cancer = sample.loc[cancer_mask].copy()
not_cancer = sample.loc[~cancer_mask].copy()
normal = sample.loc[normal_mask].copy()

summary_line(
    f"Cancer-like disease label: {len(cancer):,} SRX rows, "
    f"{cancer['disease'].nunique():,} distinct disease strings, "
    f"{int(cancer['obs_count'].sum()):,} cells (obs_count sum)"
)
normal["disease"].value_counts(normalize=True).head(40)

# No cancer labels seem to be present in the data
not_cancer["disease"].value_counts(normalize=True).head(40)

Cancer-like disease label: 9,309 SRX rows, 1,577 distinct disease strings, 80,240,424 cells (obs_count sum)


disease
none reported                                        0.042127
COVID-19                                             0.034564
normal                                               0.026395
other                                                0.021479
healthy                                              0.020572
none (healthy donor)                                 0.019437
none specified                                       0.018832
none (healthy donors)                                0.018152
control                                              0.012101
Crohn's disease                                      0.010815
none (healthy)                                       0.009076
healthy donor                                        0.007185
cardiovascular disease risk factors                  0.006958
Alzheimer's Disease                                  0.006126
multiple sclerosis                                   0.006126
SARS-CoV-2 infection                                 0.005975


## 4. Immune-oncology (IO) candidate SRX rows

Heuristic: flag rows where **any** of `perturbation`, `disease`, or `tissue` matches IO / checkpoint / CAR-T style language. This is intentionally broad for **dataset targeting**; tighten keywords for stricter cohorts.

In [11]:
IO_RE = re.compile(
    r"immuno|checkpoint|PD-1|PD1|PD-L1|PDL1|pembrolizumab|nivolumab|atezolizumab|"
    r"durvalumab|ipilimumab|CTLA-4|CTLA4|CAR[- ]?T|CART|anti-PD|ICB|KEYTRUDA|OPDIVO|"
    r"immune checkpoint|checkpoint inhibitor",
    re.I,
)

io_any = (
    sample["perturbation"].str.contains(IO_RE, na=False)
    | sample["disease"].str.contains(IO_RE, na=False)
    | sample["tissue"].str.contains(IO_RE, na=False)
)
io = sample.loc[io_any].copy()
not_io = sample.loc[~io_any].copy()
summary_line(
    f"IO keyword hit (any column): {len(io):,} SRX rows, "
    f"{int(io['obs_count'].sum()):,} cells"
)
summary_line("By column (rows can overlap):")
for col in ["perturbation", "disease", "tissue"]:
    m = sample[col].str.contains(IO_RE, na=False)
    sub = sample.loc[m]
    summary_line(
        f"  {col}: {len(sub):,} SRX, {int(sub['obs_count'].sum()):,} cells, "
        f"{sub[col].nunique():,} distinct {col} strings"
    )

IO keyword hit (any column): 1,742 SRX rows, 12,569,121 cells
By column (rows can overlap):
  perturbation: 1,607 SRX, 11,651,752 cells, 758 distinct perturbation strings
  disease: 208 SRX, 1,532,319 cells, 80 distinct disease strings
  tissue: 87 SRX, 852,035 cells, 51 distinct tissue strings


In [12]:
not_io[["perturbation", "disease", "tissue"]].value_counts(normalize=True).head(40)

perturbation                               disease                                    tissue               
day 7 post transduction                    acute T cell leukemia                      acute T cell leukemia    0.004329
CRISPR interference (CRISPRi)              chronic myelogenous leukemia               K562 cell line           0.003800
unsure                                     lung adenocarcinoma                        lung                     0.003463
                                           breast cancer                              breast                   0.001780
oxygen treatment                           COVID-19                                   blood                    0.001780
unsure                                     ocular hypertension and risk for glaucoma  trabecular meshwork      0.001732
CRISPR interference                        chronic myelogenous leukemia               K562 cell line           0.001684
unsure                                     esophagea

In [13]:
io["io_via"] = ""
io.loc[io["perturbation"].str.contains(IO_RE, na=False), "io_via"] += "perturbation;"
io.loc[io["disease"].str.contains(IO_RE, na=False), "io_via"] += "disease;"
io.loc[io["tissue"].str.contains(IO_RE, na=False), "io_via"] += "tissue;"

card_cols = [
    "srx_accession",
    "disease",
    "perturbation",
    "tissue",
    "cell_line",
    "lib_prep",
    "obs_count",
    "io_via",
    "file_path",
]
card_cols = [c for c in card_cols if c in io.columns]
io.sort_values("obs_count", ascending=False).head(25)[card_cols]

,srx_accession,disease,perturbation,tissue,cell_line,lib_prep,obs_count,io_via,file_path
6117,SRX15016115,bone marrow failure after CD19 CAR-T for Richt...,CD19-directed CAR T-cell therapy (tisagenlecle...,peripheral blood,other,10x_Genomics,146884,perturbation;disease;,gs://arc-institute-virtual-cell-atlas/scbaseco...
13786,SRX22671373,disease stage IV,Nivolumab (Nivo),blood and tumor,live CD45+ immune cells,10x_Genomics,118050,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
3269,SRX22671374,advanced melanoma,combination therapy with relatimab and nivolumab,tumor tissue and blood,not applicable,10x_Genomics,114156,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
26162,SRX15016116,bone marrow failure and DLBCL,CD19 CAR-T therapy (tisagenlecleucel),peripheral blood,not specified; sample derived from a patient,10x_Genomics,102711,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
13517,SRX23973422,acute lymphocytic leukemia,"CAR T therapy, IL-4 addition",PBMCs,CART_ADT_T,10x_Genomics,102039,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
21638,SRX10414128,muscle-invasive bladder cancer,treatment-naïve (no prior neoadjuvant chemothe...,muscle-invasive bladder cancer tumor,not_applicable,10x_Genomics,65242,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
28294,SRX27733857,Head and neck squamous cell carcinoma (HNSCC),"Immunotherapy regimens including anti-PD-1, an...",Tumor tissue from head and neck,No established cell line; primary immune cells...,10x_Genomics,55277,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
14812,SRX26487601,non-small cell lung cancer,nivolumab+chemotherapy,adjacent normal lung,unsure,10x_Genomics,52229,perturbation;,gs://arc-institute-virtual-cell-atlas/scbaseco...
27791,SRX27409352,checkpoint inhibitor-induced liver injury (ChILI),Ipilimumab + Nivolumab treatment,peripheral blood mononuclear cells (PBMCs),none,10x_Genomics,45806,perturbation;disease;,gs://arc-institute-virtual-cell-atlas/scbaseco...
33836,SRX28141444,none specified,ethanol,fecal immunochemical test (FIT) sample,none specified,10x_Genomics,40125,tissue;,gs://arc-institute-virtual-cell-atlas/scbaseco...


### IO ∩ cancer (prioritize oncology-relevant IO cohorts)

In [14]:
io_cancer = io.loc[io["disease"].str.contains(CANCER_RE, na=False) | io["tissue"].str.contains(CANCER_RE, na=False)].copy()
summary_line(
    f"IO + cancer-like disease or tissue: {len(io_cancer):,} SRX, "
    f"{int(io_cancer['obs_count'].sum()):,} cells"
)
io_cancer.groupby("disease", dropna=False)["obs_count"].sum().sort_values(ascending=False).head(25)

IO + cancer-like disease or tissue: 1,386 SRX, 9,753,666 cells


disease
multiple myeloma                                 1154158
Head and neck squamous cell carcinoma (HNSCC)     427893
melanoma                                          426627
Multiple Myeloma, early relapse                   420943
multiple myeloma, early relapse                   397065
endometrial carcinoma                             392003
Multiple Myeloma                                  329530
advanced melanoma                                 249631
Head and neck squamous cell carcinoma             241144
head and neck squamous cell carcinoma             207437
cancer                                            175636
hepatocellular carcinoma                          174340
cancer immunotherapy                              145271
follicular lymphoma                               144690
Multiple Myeloma, Early relapse                   140020
Neuroblastoma                                     130763
disease stage IV                                  123178
non-small cell lung can

### IO cohorts: cells and SRX counts per disease / collection

In this metadata, each **SRX accession** is one processed experiment (Arc’s unit linking `h5ad` and `obs_metadata` rows). It is **not** the same as individual human donors; patient *n* usually requires GEO/SRA sample attributes outside this parquet.

In [15]:
def cohort_summary(df: pd.DataFrame, key: str) -> pd.DataFrame:
    g = df.groupby(key, dropna=False).agg(
        n_srx=("srx_accession", "nunique"),
        n_cells=("obs_count", "sum"),
    )
    return g.sort_values("n_cells", ascending=False)


cohort_summary(io, "disease").head(20)

,n_srx,n_cells
disease,,
multiple myeloma,119,1154158
Head and neck squamous cell carcinoma (HNSCC),70,427893
melanoma,59,426627
"Multiple Myeloma, early relapse",40,420943
"multiple myeloma, early relapse",45,397065
endometrial carcinoma,61,392003
Multiple Myeloma,36,329530
Checkpoint inhibitor-induced liver injury (ChILI),45,287952
advanced melanoma,35,249631


In [16]:
sub = io.dropna(subset=["czi_collection_name"])
if len(sub):
    cohort_summary(sub, "czi_collection_name").head(20)
else:
    summary_line("czi_collection_name all null for IO slice in this snapshot.")

## 5. Per-SRX "data card" helper (disease, treatment modality, tissue, size)

Use `format_study_card` for **one SRX row**. For **cohort-level** IO summaries by disease area (many SRX, cells + labels), use **section 5b** below.

In [17]:
def format_study_card(row: pd.Series) -> str:
    return (
        f"SRX: {row.get('srx_accession', '')}\n"
        f"Disease: {row.get('disease', '')}\n"
        f"Tissue: {row.get('tissue', '')}\n"
        f"Treatment / perturbation: {row.get('perturbation', '')}\n"
        f"Cell line: {row.get('cell_line', '')}\n"
        f"Library prep: {row.get('lib_prep', '')}\n"
        f"Cells (obs_count): {int(row.get('obs_count', 0) or 0):,}\n"
        f"h5ad path: {row.get('file_path', '')}"
    )


example = io.sort_values("obs_count", ascending=False).iloc[0]
summary_line(
    f"example IO study card (largest obs_count): SRX={example.get('srx_accession', '')}, "
    f"cells={int(example.get('obs_count', 0) or 0):,}"
)
print(format_study_card(example))

example IO study card (largest obs_count): SRX=SRX15016115, cells=146,884
SRX: SRX15016115
Disease: bone marrow failure after CD19 CAR-T for Richter transformed DLBCL
Tissue: peripheral blood
Treatment / perturbation: CD19-directed CAR T-cell therapy (tisagenlecleucel)
Cell line: other
Library prep: 10x_Genomics
Cells (obs_count): 146,884
h5ad path: gs://arc-institute-virtual-cell-atlas/scbasecount/2026-01-12/h5ad/GeneFull/Homo_sapiens/SRX15016115.h5ad


## 5b. IO × disease-theme cohort cards (cells + SRX)

Repeat the same pattern for each **disease theme**: intersect the IO keyword slice with rows whose **`disease` or `tissue`** text matches a theme regex. Summarize **SRX count**, **cell count** (`obs_count` sum), and **distinct raw `disease` strings** (subtype richness in label space—not clinical staging).

Edit **`DISEASE_THEMES`** to add or narrow patterns (e.g. hematologic vs solid). Optional **`THEME_SUBTYPE_RULES`** buckets raw labels into a short list for card text (e.g. ER+/HER2+ style groupings when you define regexes yourself).

In [18]:
DISEASE_THEMES: dict[str, re.Pattern[str]] = {
    "breast": re.compile(
        r"breast|mammary|DCIS|ductal carcinoma|invasive ductal|invasive lobular|lobular"
        r"|HER2|triple[- ]negative|ER[+-]|PR[+-]"
        r"|inflammatory breast|Paget.s disease"
        r"|BRCA",
        re.I,
    ),
    "lung": re.compile(
        r"\blung\b|pulmonary|NSCLC|SCLC|adenocarcinoma of the lung|mesothelioma|pleura"
        r"|squamous cell carcinoma of the lung|large[- ]cell|bronch"
        r"|EGFR|(?<!\w)ALK(?!\w)|ROS1"
        r"|thymoma|thymic",
        re.I,
    ),
    "melanoma": re.compile(
        r"melanoma|cutaneous melanoma|uveal melanoma|mucosal melanoma|\bBRAF\b",
        re.I,
    ),
    "endometrial": re.compile(
        r"endometrial|endometrium|\buterine\b|uterus|corpus\s+uteri|endometrioid"
        r"|\bserous carcinoma\b|\bclear[- ]cell\b",
        re.I,
    ),
    "cns": re.compile(
        r"glioblastoma|glioma|astrocytoma|oligodendroglioma|ependymoma|medulloblastoma"
        r"|pineoblastoma|CNS lymphoma|primary CNS"
        r"|\bCNS\b|intracranial|brain metastases|brain metastasis|brain tumor|brain tumour"
        r"|spinal cord tumor|spinal cord tumour|cerebellar tumor|pituitary adenoma",
        re.I,
    ),
    "hepatobiliary": re.compile(
        r"hepatocellular|\bHCC\b|cholangiocarcinoma|cholangio|bile duct|biliary"
        r"|gallbladder|intrahepatic|extrahepatic cholangio|liver cancer|hepatic cancer|primary liver",
        re.I,
    ),
    "gi": re.compile(
        r"colorectal|colon cancer|rectal cancer|rectal adenocarcinoma|\bCRC\b"
        r"|gastric|stomach cancer|gastric adenocarcinoma"
        r"|esophag|oesophag|pancreatic|pancreas cancer|pancreatic adenocarcinoma"
        r"|small intestine|duodenal cancer|appendiceal|anal cancer|anal squamous",
        re.I,
    ),
    "genitourinary": re.compile(
        r"bladder|urothelial|transitional cell"
        r"|renal cell|kidney cancer|\bRCC\b"
        r"|prostate cancer|prostatic adenocarcinoma|prostate adenocarcinoma"
        r"|ureter|urethral cancer|testicular|seminoma|embryonal carcinoma",
        re.I,
    ),
    "ovarian_cervical": re.compile(
        r"ovarian|ovary|fallopian|tubo-ovarian"
        r"|cervical cancer|cervix cancer|cervical carcinoma|cervical squamous"
        r"|vulvar|vulva|vaginal cancer|vaginal carcinoma"
        r"|primary peritoneal|peritoneal serous",
        re.I,
    ),
    "skin_non_melanoma": re.compile(
        r"Merkel cell|basal cell carcinoma|cutaneous squamous"
        r"|squamous cell carcinoma of the skin|non-melanoma skin",
        re.I,
    ),
    "sarcoma": re.compile(
        r"osteosarcoma|chondrosarcoma|ewing|ewing.s sarcoma|soft tissue sarcoma"
        r"|rhabdomyosarcoma|synovial sarcoma|liposarcoma|leiomyosarcoma|angiosarcoma"
        r"|undifferentiated sarcoma|\bsarcoma\b",
        re.I,
    ),
    "hematologic": re.compile(
        r"leukemia|leukaemia|lymphoma|myeloma|myeloid|\bCML\b|\bAML\b"
        r"|\bCLL\b|DLBCL|multiple myeloma"
        r"|lymphocytic|\bHodgkin\b|non[- ]Hodgkin"
        r"|\bMDS\b|myelodysplastic|plasma cell"
        r"|follicular lymphoma|mantle cell|Burkitt"
        r"|T[- ]cell lymphoma|B[- ]cell lymphoma"
        r"|\bblast\b",
        re.I,
    ),
    "head_and_neck": re.compile(
        r"head and neck|HNSCC|oral cavity|oropharyn|nasopharyn|hypopharyn|laryngeal"
        r"|thyroid|salivary gland|parotid"
        r"|sinonasal|nasal cavity|\bsinus\b"
        r"|glottis|supraglottic",
        re.I,
    ),
}


def disease_or_tissue_matches(pat: re.Pattern[str], frame: pd.DataFrame) -> pd.Series:
    d = frame["disease"].fillna("").astype(str)
    t = frame["tissue"].fillna("").astype(str)
    return d.str.contains(pat, na=False) | t.str.contains(pat, na=False)


def io_theme_cohort(
    sample_df: pd.DataFrame, io_mask: pd.Series, theme_pat: re.Pattern[str]
) -> pd.DataFrame:
    base = sample_df.loc[io_mask].copy()
    return base.loc[disease_or_tissue_matches(theme_pat, base)]


def format_io_theme_card(
    theme_name: str,
    cohort: pd.DataFrame,
    *,
    top_disease: int = 8,
    top_perturbation: int = 6,
    top_tissue: int = 5,
) -> str:
    if cohort.empty:
        return f"Theme: {theme_name}\n(no rows in IO ∩ theme slice)"

    n_srx = cohort["srx_accession"].nunique()
    w = pd.to_numeric(cohort["obs_count"], errors="coerce").fillna(0)
    n_cells = int(w.sum())
    n_labels = cohort["disease"].nunique()

    lines = [
        f"Theme: {theme_name}",
        f"In this IO-filtered slice we have {n_srx:,} SRX experiments and {n_cells:,} cells (obs_count sum).",
        f"Distinct raw disease strings: {n_labels:,} (label heterogeneity in metadata).",
        "",
        "Top disease labels by cell count:",
    ]
    g = cohort.groupby("disease", dropna=False)["obs_count"].sum().sort_values(ascending=False)
    for lab, cells in g.head(top_disease).items():
        lines.append(f"  - {lab!s}: {int(cells):,} cells")

    lines.append("")
    lines.append("Top perturbation labels by cell count:")
    g2 = cohort.groupby("perturbation", dropna=False)["obs_count"].sum().sort_values(ascending=False)
    for lab, cells in g2.head(top_perturbation).items():
        lines.append(f"  - {lab!s}: {int(cells):,} cells")

    lines.append("")
    lines.append("Top tissue labels by cell count:")
    g3 = cohort.groupby("tissue", dropna=False)["obs_count"].sum().sort_values(ascending=False)
    for lab, cells in g3.head(top_tissue).items():
        lines.append(f"  - {lab!s}: {int(cells):,} cells")

    return "\n".join(lines)


# Optional: bucket raw disease text into a short list for narrative cards (patterns are yours to refine).
THEME_SUBTYPE_RULES: dict[str, list[tuple[re.Pattern[str], str]]] = {
    "breast": [
        (re.compile(r"triple[- ]negative|TNBC", re.I), "triple_negative"),
        (re.compile(r"HER2|ERBB2", re.I), "HER2_mentioned"),
        (re.compile(r"ER\+|estrogen receptor positive|ER positive", re.I), "ER_positive_mentioned"),
        (re.compile(r"PR\+|progesterone receptor positive", re.I), "PR_positive_mentioned"),
    ],
}


def subtype_bucket_cell_counts(
    cohort: pd.DataFrame, rules: list[tuple[re.Pattern[str], str]]
) -> pd.Series:
    w = pd.to_numeric(cohort["obs_count"], errors="coerce").fillna(0)
    s = cohort["disease"].fillna("").astype(str)
    out: dict[str, int] = {}
    for pat, name in rules:
        out[name] = int(w[s.str.contains(pat, na=False)].sum())
    return pd.Series(out, dtype="int64")


cohorts: dict[str, pd.DataFrame] = {
    name: io_theme_cohort(sample, io_any, pat) for name, pat in DISEASE_THEMES.items()
}

overview_rows = []
for theme_name, ch in cohorts.items():
    w = pd.to_numeric(ch["obs_count"], errors="coerce").fillna(0)
    overview_rows.append(
        {
            "theme": theme_name,
            "n_srx": ch["srx_accession"].nunique(),
            "n_cells": int(w.sum()),
            "n_distinct_disease": ch["disease"].nunique(),
        }
    )

theme_overview = pd.DataFrame(overview_rows).sort_values("n_cells", ascending=False)
for _, r in theme_overview.iterrows():
    summary_line(
        f"IO∩theme overview {r['theme']!r}: {int(r['n_srx']):,} SRX, "
        f"{int(r['n_cells']):,} cells, {int(r['n_distinct_disease']):,} distinct disease strings"
    )
theme_overview

IO∩theme overview 'hematologic': 502 SRX, 4,232,776 cells, 93 distinct disease strings
IO∩theme overview 'head_and_neck': 185 SRX, 1,051,736 cells, 14 distinct disease strings
IO∩theme overview 'melanoma': 174 SRX, 973,031 cells, 31 distinct disease strings
IO∩theme overview 'lung': 98 SRX, 870,336 cells, 43 distinct disease strings
IO∩theme overview 'cns': 96 SRX, 632,732 cells, 50 distinct disease strings
IO∩theme overview 'endometrial': 98 SRX, 623,612 cells, 24 distinct disease strings
IO∩theme overview 'hepatobiliary': 64 SRX, 402,593 cells, 12 distinct disease strings
IO∩theme overview 'breast': 43 SRX, 215,312 cells, 15 distinct disease strings
IO∩theme overview 'ovarian_cervical': 26 SRX, 122,455 cells, 7 distinct disease strings
IO∩theme overview 'genitourinary': 7 SRX, 96,339 cells, 3 distinct disease strings
IO∩theme overview 'sarcoma': 12 SRX, 87,085 cells, 4 distinct disease strings
IO∩theme overview 'skin_non_melanoma': 15 SRX, 45,695 cells, 2 distinct disease strings
IO∩

,theme,n_srx,n_cells,n_distinct_disease
11,hematologic,502,4232776,93
12,head_and_neck,185,1051736,14
2,melanoma,174,973031,31
1,lung,98,870336,43
4,cns,96,632732,50
3,endometrial,98,623612,24
5,hepatobiliary,64,402593,12
0,breast,43,215312,15
8,ovarian_cervical,26,122455,7
7,genitourinary,7,96339,3


### IO + cancer outside `DISEASE_THEMES`

Within **IO ∩ cancer-like `disease` or `tissue`** (`io_cancer` above), this flags SRX rows where **neither `disease` nor `tissue`** matches any pattern in `DISEASE_THEMES` (same rule as the theme cohort cards). Common reasons: IO signal is mostly in **`perturbation`**, or disease/tissue wording does not hit the current theme regexes.


In [19]:
theme_any = pd.Series(False, index=io_cancer.index)
for pat in DISEASE_THEMES.values():
    theme_any |= disease_or_tissue_matches(pat, io_cancer)

io_cancer_no_theme = io_cancer.loc[~theme_any].copy()
_w = pd.to_numeric(io_cancer_no_theme["obs_count"], errors="coerce").fillna(0)
_total_w = pd.to_numeric(io_cancer["obs_count"], errors="coerce").fillna(0)

summary_line(
    f"IO+cancer outside any DISEASE_THEMES (disease/tissue): "
    f"{len(io_cancer_no_theme):,} SRX rows, {int(_w.sum()):,} cells "
    f"of {len(io_cancer):,} SRX / {int(_total_w.sum()):,} cells in io_cancer"
)

if io_cancer_no_theme.empty:
    print("All io_cancer rows match at least one theme on disease or tissue.")
else:
    print("Top labels by cell count (obs_count) in this leftover slice:")
    display(summarize_labels(io_cancer_no_theme, "disease", top_n=30))
    display(summarize_labels(io_cancer_no_theme, "tissue", top_n=30))
    display(summarize_labels(io_cancer_no_theme, "perturbation", top_n=30))


IO+cancer outside any DISEASE_THEMES (disease/tissue): 117 SRX rows, 866,067 cells of 1,386 SRX / 9,753,666 cells in io_cancer
Top labels by cell count (obs_count) in this leftover slice:


,cells,n_srx_rows
disease,,
Neuroblastoma,130763,14
disease stage IV,123178,4
neuroblastoma,75070,5
cancer,68030,11
Checkpoint inhibitor-induced liver injury (ChILI) in cancer patients,45169,5
Cancer,29816,4
stage IV,22746,12
Cancer patients undergoing anti-PD-1 treatment,22729,2
"stage IV, female",17843,4


,cells,n_srx_rows
tissue,,
blood and tumor,123771,4
peripheral blood,77416,5
tumor,69296,29
Peripheral Blood Mononuclear Cells (PBMCs),46256,3
T cells,42370,5
Peripheral blood,35336,3
T cells/tumor,33696,6
CAR T cells,33626,2
Peripheral blood mononuclear cells (PBMCs),23157,4


,cells,n_srx_rows
perturbation,,
Nivolumab (Nivo),125163,5
CAR-T products,60842,4
"PD1-TIM3, PD1-LAG3, anti-PD1, control isotype",51898,7
GD2-directed CAR-T cell therapy,35336,3
CAR T therapy,35115,5
Immune checkpoint inhibitor therapy,24738,1
CAR-T cell therapy,24429,2
nivolumab,22072,8
Anti-PD-1 immunotherapy,18128,1


### Theme check against raw labels

**Coverage:** how many SRX rows match **0 / 1 / 2+** theme regexes (on `disease` **or** `tissue`)? Rows can match multiple themes if strings fit several patterns.

**Per theme:** SRX and cell totals on the **full** sample (same as above, **not** IO-filtered)—compare to `theme_overview`, which is **IO ∩ theme**.

**Top diseases inside each theme** helps you tighten or split regexes (e.g. move borderline strings into a new theme).

In [20]:
theme_match_matrix = np.column_stack(
    [disease_or_tissue_matches(pat, sample).to_numpy() for pat in DISEASE_THEMES.values()]
)
n_hit = theme_match_matrix.sum(axis=1)
hit_dist = pd.Series(n_hit).value_counts().sort_index()
hit_dist.index.name = "n_themes_matched"
hit_dist.name = "n_srx_rows"
summary_line(
    "How many theme patterns match each SRX row (disease OR tissue text): "
    + "; ".join(f"{int(k)} themes -> {int(v):,} SRX rows" for k, v in hit_dist.items()),
    also_print=False
)
display(hit_dist.to_frame())

rows_full = []
for name, pat in DISEASE_THEMES.items():
    m = disease_or_tissue_matches(pat, sample)
    sub = sample.loc[m]
    w = pd.to_numeric(sub["obs_count"], errors="coerce").fillna(0)
    rows_full.append(
        {"theme": name, "n_srx": len(sub), "n_cells": int(w.sum())}
    )
theme_on_full_sample = pd.DataFrame(rows_full).sort_values("n_cells", ascending=False)
for _, r in theme_on_full_sample.iterrows():
    summary_line(
        f"full-sample theme {r['theme']!r}: {int(r['n_srx']):,} SRX, {int(r['n_cells']):,} cells",
        also_print=False
    )
display(theme_on_full_sample)

TOP_DISEASES_PER_THEME = 12
for name, pat in DISEASE_THEMES.items():
    sub = sample.loc[disease_or_tissue_matches(pat, sample)]
    if sub.empty:
        print(f"--- theme {name!r}: (no rows) ---")
        continue
    g = (
        sub.groupby("disease", dropna=False)["obs_count"]
        .sum()
        .sort_values(ascending=False)
        .head(TOP_DISEASES_PER_THEME)
    )
    print(f"--- theme {name!r}: top diseases by cells ---")
    display(g.astype(int).to_frame("cells"))

,n_srx_rows
n_themes_matched,
0,11876
1,10457
2,193
3,4
4,1


,theme,n_srx,n_cells
11,hematologic,4036,45755145
1,lung,1859,11309348
0,breast,1047,7721470
6,gi,1069,6579864
4,cns,601,3718503
8,ovarian_cervical,459,2681664
5,hepatobiliary,332,2404831
7,genitourinary,291,2246084
2,melanoma,388,2172391
12,head_and_neck,387,2003300


--- theme 'breast': top diseases by cells ---


,cells
disease,
breast cancer,1983288
normal,719262
Hormone dependent breast cancer (HDBC),312471
triple-negative breast cancer,264462
hormone-dependent breast cancer,192351
hormone dependent breast cancer,164984
triple-negative breast cancer (TNBC),153518
Breast cancer,131719
invasive breast carcinoma,128685


--- theme 'lung': top diseases by cells ---


...

,cells
disease,
osteosarcoma,143133
pediatric rhabdomyosarcoma,130834
rhabdomyosarcoma,112411
Rhabdomyosarcoma,73860
Rhabdomyosarcoma (RMS),53677
Dedifferentiated Liposarcoma,46474
leiomyosarcoma,42686
dedifferentiated liposarcoma,38601
Kaposi's sarcoma-associated herpesvirus (KSHV) infection,37786


--- theme 'hematologic': top diseases by cells ---


,cells
disease,
chronic myelogenous leukemia,13421381
acute T cell leukemia,5550486
Chronic Myelogenous Leukemia,4336204
Chronic Myelogenous Leukemia (CML),1968445
multiple myeloma,1618395
Acute T cell leukemia,1505159
chronic myelogenous leukemia (CML),1101162
Chronic Myeloid Leukemia,999017
Chronic myelogenous leukemia (CML),917771


--- theme 'head_and_neck': top diseases by cells ---


,cells
disease,
Head and neck squamous cell carcinoma (HNSCC),457052
head and neck squamous cell carcinoma,262334
Head and neck squamous cell carcinoma,246636
head and neck squamous cell carcinoma (HNSCC),114110
HPV-positive head and neck squamous cell carcinoma,89420
thyroid carcinoma,47759
papillary thyroid cancer,42308
liver metastatic breast cancer,33497
Head and Neck Squamous Cell Carcinoma (HNSCC),31403


In [21]:
for theme in theme_overview.loc[theme_overview["n_cells"] > 0, "theme"]:
    ch = cohorts[theme]
    if ch.empty:
        summary_line(f"IO∩theme {theme!r}: (no rows)")
    else:
        _w = pd.to_numeric(ch["obs_count"], errors="coerce").fillna(0)
        summary_line(
            f"IO∩theme {theme!r}: {ch['srx_accession'].nunique():,} SRX, "
            f"{int(_w.sum()):,} cells, {ch['disease'].nunique():,} distinct disease strings"
        )
    print(format_io_theme_card(theme, ch))
    print("=" * 72)

for theme, rules in THEME_SUBTYPE_RULES.items():
    ch = cohorts[theme]
    if ch.empty:
        continue
    bc = subtype_bucket_cell_counts(ch, rules)
    if bc.sum() == 0:
        continue
    summary_line(
        f"Optional disease-text buckets for theme {theme!r}: "
        + ", ".join(f"{k}={int(v):,}" for k, v in bc[bc > 0].items())
    )
    print(f"\nOptional disease-text buckets for theme {theme!r} (cells; rows can match multiple patterns):")
    print(bc[bc > 0].to_string())

IO∩theme 'hematologic': 502 SRX, 4,232,776 cells, 93 distinct disease strings
Theme: hematologic
In this IO-filtered slice we have 502 SRX experiments and 4,232,776 cells (obs_count sum).
Distinct raw disease strings: 93 (label heterogeneity in metadata).

Top disease labels by cell count:
  - multiple myeloma: 1,154,158 cells
  - Multiple Myeloma, early relapse: 420,943 cells
  - multiple myeloma, early relapse: 397,065 cells
  - Multiple Myeloma: 329,530 cells
  - bone marrow failure after CD19 CAR-T for Richter transformed DLBCL: 146,884 cells
  - follicular lymphoma: 144,690 cells
  - Multiple Myeloma, Early relapse: 140,020 cells
  - bone marrow failure and DLBCL: 102,711 cells

Top perturbation labels by cell count:
  - BCMA CAR-T cell therapy: 1,945,017 cells
  - BCMA CAR-T therapy: 216,583 cells
  - CAR T therapy, IL-4 addition: 164,856 cells
  - CD19-directed CAR T-cell therapy (tisagenlecleucel): 146,884 cells
  - CD19 CAR-T therapy (tisagenlecleucel): 102,711 cells
  - RCHOP

## 6. Patient / stage / progression fields — coverage in these parquets

`sample_metadata` exposes study-level disease, tissue, perturbation, and technology fields. There is **no** dedicated patient ID or cancer **stage** column in the schema we loaded.

Optional: scan string fields for words like `stage`, `grade`, `TNM`, `metastatic` to see if they appear inside free-text labels.

In [22]:
STAGE_RE = re.compile(
    r"stage\s*[IV0-4]|TNM|metastatic|metastasis|grade\s*[0-4]|AJCC",
    re.I,
)

text_cols = [c for c in sample.columns if sample[c].dtype == object]
hits = []
for c in text_cols:
    m = sample[c].fillna("").astype(str).str.contains(STAGE_RE, na=False)
    if m.any():
        hits.append((c, int(m.sum())))
hits = sorted(hits, key=lambda x: -x[1])
summary_line("Column -> rows with stage-like tokens:")
for c, k in hits[:15]:
    summary_line(f"  {c}: {k}")
if not hits:
    summary_line("  (none)")

Column -> rows with stage-like tokens:
  (none)


In [23]:
if hits:
    c = hits[0][0]
    m = sample[c].fillna("").astype(str).str.contains(STAGE_RE, na=False)
    sample.loc[m, c].value_counts().head(15)

## 7. Optional: cell types for a **small** SRX set via `obs_metadata`

The Homo sapiens `obs_metadata` parquet can be hundreds of millions of rows. The cell below uses a PyArrow **dataset filter** on `SRX_accession` so only matching rows are materialized (runtime still depends on how well row-group statistics prune the scan).

For heavier summaries, consider DuckDB `read_parquet(..., hive_partitioning=...)` or Polars lazy scans.

In [24]:
def obs_rows_for_srx(
    srx_accession: str,
    *,
    parquet_path: Path | None = None,
    columns: list[str] | None = None,
) -> pd.DataFrame:
    """One SRX from obs_metadata: filter pushed into the parquet scan (no full-file load)."""
    dset = ds.dataset(str(parquet_path or OBS_PARQUET), format="parquet")
    tbl = dset.to_table(
        columns=columns,
        filter=ds.field("SRX_accession") == srx_accession,
    )
    return tbl.to_pandas()

# obs_rows_for_srx("SRX15016115", columns=["SRX_accession", "cell_barcode", "cell_type"])

In [25]:
obs_rows_for_srx("SRX23892534")["cell_type"].value_counts(normalize=True)

cell_type
leukocyte          0.989661
oligodendrocyte    0.005215
                   0.003738
neuron             0.000844
astrocyte          0.000543
Name: proportion, dtype: float64

In [26]:
TOP_N = 5
top_srx = io.sort_values("obs_count", ascending=False).head(TOP_N)["srx_accession"].tolist()
top_srx

['SRX15016115', 'SRX22671373', 'SRX22671374', 'SRX15016116', 'SRX23973422']

In [27]:
use_cols = ["SRX_accession", "cell_type", "cell_ontology_term_id"]
if not top_srx:
    summary_line("No IO rows selected; widen IO keywords or check data.")
else:
    dset = ds.dataset(str(OBS_PARQUET), format="parquet")
    tbl = dset.to_table(
        columns=use_cols,
        filter=ds.field("SRX_accession").isin(pa.array(top_srx)),
    )
    obs_sub = tbl.to_pandas()
    summary_line(f"Loaded {len(obs_sub):,} cell rows across {len(top_srx)} SRX accessions")
    (
        obs_sub.groupby(["SRX_accession", "cell_type"], dropna=False)
        .size()
        .sort_values(ascending=False)
        .head(35)
    )

Loaded 583,840 cell rows across 5 SRX accessions


In [28]:
sample["disease"].value_counts(normalize=True)

disease
chronic myelogenous leukemia                        0.041498
none reported                                       0.024721
acute T cell leukemia                               0.022769
COVID-19                                            0.020283
normal                                              0.015490
                                                      ...   
None (healthy donor, no ophthalmologic disease)     0.000044
dystonia                                            0.000044
none specified (normal fetal development)           0.000044
none (healthy first-trimester placenta)             0.000044
unsure (likely ovarian cancer, but not explicit)    0.000044
Name: proportion, Length: 5604, dtype: float64

# strict filtering

In [29]:
sample = pd.read_parquet(SAMPLE_PARQUET)

IO_RE
DISEASE_THEMES
_UNKNOWN_LIKE_DISEASE = re.compile(r'\b(?:\bunknown\b|\bunsure\b|\bunclear\b|\bnot reported\b|\bnot specified\b|\bnone reported\b|\bnone specified\b|\bnone\b|\bno disease\b|\bno diagnosis\b|\bhealthy\b|\bnormal\b|\bno treatment\b|\bcontrol\b|\bnon-diseased\b|\bnot applicable\b|\bna)\b', re.IGNORECASE)
CANCER_RE

re.compile(r'cancer|carcinoma|melanoma|lymphoma|leukemia|leukaemia|myeloma|sarcoma|tumor|tumour|neoplasm|oma\b|glioma|blastoma',
           re.IGNORECASE|re.UNICODE)

In [ ]:
unknown_mask = sample[["disease", "tissue"]].apply(lambda col: col.str.contains(_UNKNOWN_LIKE_DISEASE, na=False)).any(axis=1)
sample_known = sample.loc[unknown_mask == False]

cancer_mask = sample_known[["disease"]].apply(lambda col: col.str.contains(CANCER_RE, na=False)).any(axis=1)
sample_cancer = sample_known.loc[cancer_mask]

io_mask = sample_cancer[["disease", "tissue", "perturbation"]].apply(lambda col: col.str.contains(IO_RE, na=False)).any(axis=1)
sample_io = sample_cancer.loc[io_mask]

import re

# Create a function to extract disease theme for each row
def extract_theme(row: pd.Series) -> str:
    # Try to find first matching theme in either disease or tissue column
    disease = str(row["disease"]) if not pd.isnull(row["disease"]) else ""
    tissue = str(row["tissue"]) if not pd.isnull(row["tissue"]) else ""
    for theme, pat in DISEASE_THEMES.items():
        if re.search(pat, disease) or re.search(pat, tissue):
            return theme
    return "none"

# Apply the extraction function to the sample_io DataFrame
sample_io["theme"] = sample_io.apply(extract_theme, axis=1)

# Groupby summary with the identified disease theme
theme_summary = (
    sample_io.groupby("theme")
    .agg(
        n_srx=("srx_accession", "nunique"),
        n_cells=("obs_count", "sum"),
        n_distinct_disease=("disease", "nunique"),
    )
    .sort_values("n_cells", ascending=False)
)
print(theme_summary)


sample_io[]

                   n_srx  n_cells  n_distinct_disease
theme                                                
hematologic          486  3896655                  85
head_and_neck        179  1003418                  11
melanoma             173   962811                  30
none                  87   666868                  45
endometrial          104   648907                  26
lung                  76   645328                  32
cns                   86   624648                  41
hepatobiliary         62   397692                  11
breast                43   215312                  15
genitourinary          9   100637                   5
sarcoma               11    71941                   3
ovarian_cervical       7    47233                   4
skin_non_melanoma     15    45695                   2
gi                     7    42867                   7


,entrez_id,srx_accession,file_path,obs_count,lib_prep,tech_10x,cell_prep,organism,tissue,tissue_ontology_term_id,disease,disease_ontology_term_id,perturbation,cell_line,antibody_derived_tag,czi_collection_id,czi_collection_name,theme
211,27937117,SRX20513536,gs://arc-institute-virtual-cell-atlas/scbaseco...,4843,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,peripheral blood mononuclear cells,UBERON:0000178,hepatocellular carcinoma,MONDO:0007256,immune checkpoint blockade,peripheral blood mononuclear cells,no,NaN,NaN,hepatobiliary
231,27937118,SRX20513537,gs://arc-institute-virtual-cell-atlas/scbaseco...,1292,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,peripheral blood mononuclear cells (PBMCs),UBERON:0000178,ICB-treated hepatocellular carcinoma (HCC),MONDO:0007256,immune checkpoint blockade (ICB),not specified,no,NaN,NaN,hepatobiliary
269,29025499,SRX21541857,gs://arc-institute-virtual-cell-atlas/scbaseco...,10218,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,bone marrow,UBERON:0002371,multiple myeloma,MONDO:0009693,CAR-T cell therapy,T cells,no,NaN,NaN,hematologic
275,28617013,SRX21181679,gs://arc-institute-virtual-cell-atlas/scbaseco...,4662,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,brain metastases,,recurrent glioblastoma,MONDO:0018177,neoadjuvant PD-1 checkpoint blockade therapy,immune cells,no,NaN,NaN,cns
282,29025249,SRX21541607,gs://arc-institute-virtual-cell-atlas/scbaseco...,34410,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,bone marrow,UBERON:0002371,multiple myeloma,MONDO:0009693,"vehicle treated, BCAR-T treated, Allo15BCAR-NK...",tumor cells (MM-FG cells),no,NaN,NaN,hematologic
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34493,11419347,SRX8784080,gs://arc-institute-virtual-cell-atlas/scbaseco...,1260,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,glioblastoma tumor-infiltrating lymphocytes an...,,glioblastoma,MONDO:0018177,neoadjuvant anti-PD-1 therapy,none,no,999f2a15-3d7e-440b-96ae-2c806799c08c,"Harmonized single-cell landscape, intercellula...",cns
34689,11419338,SRX8784071,gs://arc-institute-virtual-cell-atlas/scbaseco...,5762,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,glioblastoma (brain tumor),,glioblastoma,MONDO:0018177,neoadjuvant anti-PD-1 (PD-1 blockade),none,no,999f2a15-3d7e-440b-96ae-2c806799c08c,"Harmonized single-cell landscape, intercellula...",cns
34746,11419340,SRX8784073,gs://arc-institute-virtual-cell-atlas/scbaseco...,7084,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,glioblastoma tumor,UBERON:0000955,glioblastoma,MONDO:0018177,neoadjuvant anti-PD-1 checkpoint blockade,not_applicable,no,999f2a15-3d7e-440b-96ae-2c806799c08c,"Harmonized single-cell landscape, intercellula...",cns
34798,11419330,SRX8784063,gs://arc-institute-virtual-cell-atlas/scbaseco...,3062,10x_Genomics,3_prime_gex,single_cell,Homo sapiens,glioblastoma tumor-infiltrating lymphocytes an...,,recurrent glioblastoma,MONDO:0018177,pre-treatment with neoadjuvant anti-PD-1 therapy,none,no,999f2a15-3d7e-440b-96ae-2c806799c08c,"Harmonized single-cell landscape, intercellula...",cns


In [32]:



# obs_rows_for_srx("SRX15016115")#, columns=["cell_barcode"])
# pf_obs
pf_obs.read_row_group(0)

pyarrow.Table
cell_barcode: string
SRX_accession: string
gene_count_Unique: int64
umi_count_Unique: float
gene_count_UniqueAndMult-EM: int64
umi_count_UniqueAndMult-EM: float
gene_count_UniqueAndMult-Uniform: int64
umi_count_UniqueAndMult-Uniform: float
cell_type: string
cell_ontology_term_id: string
----
cell_barcode: [["AAACCCAAGAAGCCTG","AAACCCAAGAATCTAG","AAACCCAAGACTTAAG","AAACCCAAGACTTCAC","AAACCCAAGCCGTTGC",...,"TTCTTCCGTACAGTTC","TTCTTCCGTAGAAACT","TTCTTCCGTCCCGTGA","TTCTTCCGTGAGACGT","TTCTTCCGTTAAGCAA"]]
SRX_accession: [["ERX10019090","ERX10019090","ERX10019090","ERX10019090","ERX10019090",...,"ERX10987142","ERX10987142","ERX10987142","ERX10987142","ERX10987142"]]
gene_count_Unique: [[5980,6325,5444,2298,979,...,5702,3332,1747,1521,6632]]
umi_count_Unique: [[14205,18191,16093,4143,1189,...,33108,8677,3202,2771,32488]]
gene_count_UniqueAndMult-EM: [[6230,6665,5767,2460,1037,...,5988,3499,1884,1626,6911]]
umi_count_UniqueAndMult-EM: [[14919.995,19927.996,16919.986,4476.9995,1344